# How to Use the satComTopology Project
This tutorial is destined to people who want to create a simulation without having to modify the code. It will only use simple features.
It will use the Yaml configuration system, which is like an auto pilot. The manual version will follow.

## Install the project
### Before
Make sure you have the right. You **must** be admitted into the [GitHub Project](https://github.com/elbarbi/sat_com_topology).
If you are not, you have to ask the authorizations to a maintainer.

### Install with pip
```shell
pip uninstall sat_com_topology -y
pip install git+https://github.com/elbarbi/satComTopology.git@develop#egg=sat_com_topology
```


## Load a full topology
### Prepare your configuration files
#### Satellites
First, you will need your TLE file, a TLE file is txt that contains every satellite TLE.
If you don't know what is a TLE here is the [wikipedia page](https://en.wikipedia.org/wiki/Two-line_element_set).

Here is an example of the file that you should provide.
```txt
72 22
NewGalaxy 0
1 00001U 00000ABC 00001.00000000  .00000000  00000-0  00000+0 0    04
2 00001  53.0000   0.0000 0000000   0.0000   0.0000 15.50000000    02
NewGalaxy 1
1 00002U 00000ABC 00001.00000000  .00000000  00000-0  00000+0 0    05
2 00002  53.0000   0.0000 0000000   0.0000  16.3636 15.50000000    08
...
..
.
```
This is a fake mega-constellation, each satellite the following name: NewGalaxy {number}.
The constellation format is 72 22, which mean, **72 orbits** with **22 satellites** each.

#### Ground Stations
Ground stations are placed in a file following this format:
```txt
0,Tokyo,35.6895,139.69171,0
1,Delhi,28.66667,77.21667,0
2,Shanghai,31.22222,121.45806,0
3,São-Paulo,-23.5475,-46.63611,0
4,Mumbai-(Bombay),19.073975,72.880838,0
```
First element is the identifier, the second is the name of the city and the next are the coordinates (in degree). A ground station is associated with a city. It is easier to recognize it.

#### User Terminals
User terminals aren't implemented yet. But the file will be the same, you have to replace the name of the city by the username.

#### Organize your workspace
Make sure that those configuration are located in the same folder are very close from each other. You can put them in a configuration folder.
For this project, I created a [configuration folder](./configurations)

### The Yaml configuration File
We decided to implement our first configuration system with YAML. YAML is a file format based on indent, in opposition with JSON which use mainly brackets. Our configuration file isn't well organized yet, but we will work on it !
Here is an example of our configuration file:
```yaml
simulation_name: my_super_tutorial # The name of the simulation, keep the snake_case writting
constellation_file: configurations/file_containing_TLE.txt # The TLE file we previously saw
ground_station_file: configurations/ground_stations.txt # The ground station file
ground_station_link_range: 1000000 # In meter. The ground station range, it will be used to connect ground station to satellite
start_date: 01/01/2024 10:00:00 # Simulation start date
end_date: 01/02/2024 10:00:00 # Not used yet, but will be used when we will generate simulation snapshots
movement_model: pyorbital # The movement model is very important, this is the library that will be used to compute satellite position. PyOrbital is the default library used in this application.
distance_model: geopy # Geopy is the library used to compute distances between objects, (ground distance, so not accurate for satellites but still used).
disable_ground_station_link_preload: False # This option allow the user to deactivate the loading of ground station link. If you don't want to compute every ground station link, you can disable it, trust me it reduce the computation time.
```

## Run the simulation
Now we are going to use python.

In [22]:
from sat_com_builder.configuration_manager import YamlConfigurationManager
from sat_com_adapter.adapters import NetworkXAdapter
from sat_com_adapter.adapters import CesiumAdapter

### Load the configuration

In [23]:
yaml_configuration_manager = YamlConfigurationManager("configurations/configuration_example.yaml")

In [24]:
simulation_manager = yaml_configuration_manager.load_simulation()

### Export your configuration as networkX graph
You can export your simulation to an networkX graph:
- use it with python with the networkx api, your networkx variable is **my_graph**
- save it for later in the JSON format provided by networkx

_NB: You can still save networkx graph in other format by using networkX API._


In [25]:
networkx_adapter = NetworkXAdapter(simulation_manager, "results")

my_graph = networkx_adapter.create_full_networkx_graph()

networkx_adapter.adapt()

### Visualize your constellation with Cesium
#### Register to Cesium
For this step, you will need to use the Cesium application. Indeed, we didn't creat a whole 3d visualization software, we consume an existing product.
You have to register to get a Cesium token. Here is the [cesium token page](https://ion.cesium.com/tokens?page=1).

Cesium provide a JS library that will create web browser compatible 3d render. You will get an Html file with a lot of JavaScript inside.
sat COm Topology feature a Cesium Renderer that will allow you to create lots of different render.
#### Create your export
WHen you will have your CESIUM token, fill the following variable. If you **don't** have cesium token, you will be able to create your Html page but it display anything.

In [26]:
CESIUM_TOKEN = "<YOUR GENERATED TOKEN>"

In [27]:
cesium_adapter = CesiumAdapter(simulation_manager, "results")
len(simulation_manager.get_ground_stations_links())
renderer = cesium_adapter.build_renderer_simulation_with_links(CESIUM_TOKEN)

cesium_adapter.adapt()

You can check your [result](results) folder, html file should be there. Open it with your browser and voilà.